# Guarded Workshop Cleanup

List the configured workshop-owned assets and delete them only when `CLEANUP_WORKSHOP_ASSETS=true`.

This notebook never deletes the Azure ML workspace, compute, storage, networking, registries, identities, Key Vault, or CMK resources.

In [ ]:
from pathlib import Path
import os

from azure.ai.ml import MLClient
from azure.core.exceptions import ResourceNotFoundError
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

credential = AzureCliCredential(tenant_id=os.getenv("AZURE_TENANT_ID") or None)
ml_client = MLClient(credential, os.environ["AZURE_SUBSCRIPTION_ID"], os.environ["AZURE_RESOURCE_GROUP"], os.environ["AZUREML_WORKSPACE_NAME"])
CLEANUP = os.getenv("CLEANUP_WORKSHOP_ASSETS", "false").lower() in {"1", "true", "yes"}

assets = {
    "endpoints": [os.environ["WORKSHOP_ENDPOINT_NAME"], os.environ["H2O_ENDPOINT_NAME"]],
    "models": [(os.environ["WORKSHOP_MODEL_NAME"], os.environ["WORKSHOP_MODEL_VERSION"]), (os.environ["H2O_MODEL_NAME"], os.environ["H2O_MODEL_VERSION"])],
    "environments": [(os.environ["WORKSHOP_ENVIRONMENT_NAME"], os.environ["WORKSHOP_ENVIRONMENT_VERSION"]), (os.environ["H2O_ENVIRONMENT_NAME"], os.environ["H2O_ENVIRONMENT_VERSION"])],
    "data": [(os.environ["WORKSHOP_DATA_NAME"], os.environ["WORKSHOP_DATA_VERSION"]), (os.environ["H2O_INPUT_DATA_NAME"], os.environ["H2O_INPUT_DATA_VERSION"])],
}
print(assets)

if CLEANUP:
    for endpoint_name in assets["endpoints"]:
        try:
            ml_client.online_endpoints.begin_delete(endpoint_name).result()
            print(f"Deleted endpoint: {endpoint_name}")
        except ResourceNotFoundError:
            print(f"Endpoint absent: {endpoint_name}")
    for name, version in assets["models"]:
        try:
            ml_client.models.archive(name=name, version=version)
            print(f"Archived model: {name}:{version}")
        except ResourceNotFoundError:
            print(f"Model absent: {name}:{version}")
    for name, version in assets["environments"]:
        try:
            ml_client.environments.archive(name=name, version=version)
            print(f"Archived environment: {name}:{version}")
        except ResourceNotFoundError:
            print(f"Environment absent: {name}:{version}")
    for name, version in assets["data"]:
        try:
            ml_client.data.archive(name=name, version=version)
            print(f"Archived data: {name}:{version}")
        except ResourceNotFoundError:
            print(f"Data absent: {name}:{version}")
else:
    print("Dry run only. Set CLEANUP_WORKSHOP_ASSETS=true in workshop/.env to proceed.")

## Expected Result

With cleanup disabled, the notebook lists only configured workshop assets. With cleanup enabled, endpoints are deleted and versioned assets are archived; infrastructure remains untouched.